# Convert and Run DRO-Grasp with OpenVINO™

DRO-Grasp predicts a distance-relation matrix between robot and object point clouds. This tutorial converts the deterministic PyTorch validation graph to OpenVINO IR, validates numerical accuracy, and benchmarks inference on Intel CPU and GPU.

#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Load the PyTorch model](#Load-the-PyTorch-model)
- [Prepare representative inputs](#Prepare-representative-inputs)
- [Convert to OpenVINO](#Convert-to-OpenVINO)
- [Validate OpenVINO model accuracy](#Validate-OpenVINO-model-accuracy)
- [Run inference on Intel CPU and GPU](#Run-inference-on-Intel-CPU-and-GPU)
- [Benchmark PyTorch and OpenVINO](#Benchmark-PyTorch-and-OpenVINO)
- [Analyze accuracy and performance](#Analyze-accuracy-and-performance)
- [Reusable inference utilities](#Reusable-inference-utilities)
- [Notebook validation and tests](#Notebook-validation-and-tests)
- [Conclusions](#Conclusions)

### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/dro-grasp/dro-grasp.ipynb" />

## Prerequisites
[back to top](#Table-of-contents:)

Install the notebook dependencies. An XPU-enabled PyTorch wheel is not required: PyTorch is used on CPU to load and convert the model, while OpenVINO executes inference on CPU or GPU.

In [2]:
from pathlib import Path

import requests

if not Path("notebook_utils.py").exists():
    response = requests.get(
        "https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py",
        timeout=30,
    )
    response.raise_for_status()
    Path("notebook_utils.py").write_bytes(response.content)

from notebook_utils import pip_install

pip_install(
    "-q",
    "openvino>=2026.3.0",
    "torch>=2.7",
    "matplotlib",
    "pandas",
    "requests",
    "--extra-index-url",
    "https://download.pytorch.org/whl/cpu",
)

The next cell downloads the pinned DRO-Grasp source code and pretrained checkpoint when needed, enables notebook telemetry, and displays the Python, PyTorch, OpenVINO, and available device information for the current environment.

In [ ]:
import platform
import sys
import zipfile
from pathlib import Path

from notebook_utils import download_file

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("dro-grasp.ipynb")

SOURCE_REVISION = "e973581283d8561faba53ca1477daa4029e6c00c"
SOURCE_DIR = Path("DRO-Grasp")
MODEL_DIR = Path("model")
CHECKPOINT_ARCHIVE = Path("ckpt.zip")
CHECKPOINT_PATH = SOURCE_DIR / "ckpt/model/model_3robots.pth"
MODEL_DIR.mkdir(exist_ok=True)

if not SOURCE_DIR.exists():
    archive_path = Path(f"DRO-Grasp-{SOURCE_REVISION}.zip")
    download_file(
        f"https://github.com/mhpanah/DRO-Grasp/archive/{SOURCE_REVISION}.zip",
        filename=archive_path.name,
    )
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall()
    Path(f"DRO-Grasp-{SOURCE_REVISION}").rename(SOURCE_DIR)

if not CHECKPOINT_PATH.exists():
    download_file(
        "https://github.com/zhenyuwei2003/DRO-Grasp/releases/download/v1.0/ckpt.zip",
        filename=CHECKPOINT_ARCHIVE.name,
    )
    checkpoint_dir = SOURCE_DIR / "ckpt"
    checkpoint_dir.mkdir(exist_ok=True)
    with zipfile.ZipFile(CHECKPOINT_ARCHIVE) as archive:
        archive.extractall(checkpoint_dir)

sys.path.insert(0, str(SOURCE_DIR.resolve()))

import openvino as ov
import torch

print(f"Python: {platform.python_version()}")
print(f"PyTorch: {torch.__version__}")
print(f"OpenVINO: {ov.__version__}")
print(f"Available OpenVINO devices: {ov.Core().available_devices}")

## Load the PyTorch model
[back to top](#Table-of-contents:)

`NUM_POINTS = 32` keeps the default conversion and benchmark short. DRO-Grasp's published configuration uses 512 points. The value must be at least 32 because the encoder uses 32-neighbor KNN, and divisible by four because DRO block computation splits the point cloud into four chunks.

OpenVINO preserves dynamic point-cloud dimensions in the converted graph. A later cell validates the same IR at the published 512-point shape.

In [ ]:
from types import SimpleNamespace

from model.network import Network

from dro_grasp_helper import (
    DROGraspValidationWrapper,
    OpenVINODROGrasp,
    available_inference_devices,
    benchmark,
    comparison_metrics,
    prepare_openvino_model,
)

NUM_POINTS = 32
assert NUM_POINTS >= 32 and NUM_POINTS % 4 == 0

config = SimpleNamespace(
    emb_dim=512,
    latent_dim=64,
    pretrain=None,
    center_pc=True,
    block_computing=True,
)

network = Network(config, mode="validate")
network.load_state_dict(torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=True))
pytorch_model = DROGraspValidationWrapper(network.eval())
print(f"Loaded checkpoint: {CHECKPOINT_PATH}")

## Prepare representative inputs
[back to top](#Table-of-contents:)

DRO-Grasp consumes centered robot and object point clouds with shape `[batch, points, xyz]`. The synthetic coordinates use a `0.05` scale to represent compact point clouds in metric units and avoid a degenerate near-zero output. Validation normally samples a latent vector internally; passing it explicitly makes conversion and framework comparison deterministic. The wrapper also returns only the `dro` tensor instead of the original dictionary, whose validation-only `mu` and `logvar` entries are `None`.

In [ ]:
torch.manual_seed(42)
POINT_CLOUD_SCALE = 0.05
robot_pc = torch.randn(1, NUM_POINTS, 3, dtype=torch.float32) * POINT_CLOUD_SCALE
object_pc = torch.randn(1, NUM_POINTS, 3, dtype=torch.float32) * POINT_CLOUD_SCALE
latent = torch.randn(1, config.latent_dim, dtype=torch.float32)

with torch.no_grad():
    pytorch_output = pytorch_model(robot_pc, object_pc, latent).numpy()

assert robot_pc.shape == object_pc.shape == (1, NUM_POINTS, 3)
assert latent.shape == (1, config.latent_dim)
assert pytorch_output.shape == (1, NUM_POINTS, NUM_POINTS)
print(f"robot_pc: {tuple(robot_pc.shape)}, {robot_pc.dtype}")
print(f"object_pc: {tuple(object_pc.shape)}, {object_pc.dtype}")
print(f"latent: {tuple(latent.shape)}, {latent.dtype}")
print(f"dro: {pytorch_output.shape}, {pytorch_output.dtype}")

## Convert to OpenVINO
[back to top](#Table-of-contents:)

`ov.convert_model` traces the tensor-only wrapper with representative inputs. The converted graph receives stable input and output names and is serialized as two OpenVINO IR variants: FP32 preserves the original weights, while FP16 compresses floating-point weights to reduce model size and potentially improve inference performance.

In [ ]:
IR_PATHS = {
    "FP32": MODEL_DIR / "dro_grasp_fp32.xml",
    "FP16": MODEL_DIR / "dro_grasp_fp16.xml",
}

if not all(model_path.exists() for model_path in IR_PATHS.values()):
    converted_model = ov.convert_model(
        pytorch_model,
        example_input=(robot_pc, object_pc, latent),
    )
    prepare_openvino_model(converted_model)
    ov.save_model(converted_model, IR_PATHS["FP32"], compress_to_fp16=False)
    ov.save_model(converted_model, IR_PATHS["FP16"], compress_to_fp16=True)

ov_models = {precision: ov.Core().read_model(model_path) for precision, model_path in IR_PATHS.items()}

for precision, model in ov_models.items():
    print(f"{precision} OpenVINO IR: {IR_PATHS[precision]}")
    for model_input in model.inputs:
        print(f"  Input {model_input.any_name}: {model_input.partial_shape}, {model_input.element_type}")
    print(f"  Output {model.output(0).any_name}: {model.output(0).partial_shape}, {model.output(0).element_type}")

## Validate OpenVINO model accuracy
[back to top](#Table-of-contents:)

Both FP32 and FP16 OpenVINO models receive the same point clouds and latent input as PyTorch. Max/mean absolute error, max relative error, cosine similarity, and distance-relation agreement measure conversion fidelity. The second check runs both dynamic IR variants at the published 512-point shape.

These are numerical parity metrics, not task-level grasp success; reproducing the paper evaluation additionally requires the approximately 1 GB validation data archive and CUDA-only Isaac Gym.

In [ ]:
import numpy as np
import pandas as pd

core = ov.Core()
input_data = {
    "robot_pc": robot_pc.numpy(),
    "object_pc": object_pc.numpy(),
    "latent": latent.numpy(),
}
compiled_cpu_models = {precision: core.compile_model(model, "CPU") for precision, model in ov_models.items()}
openvino_cpu_outputs = {precision: model(input_data)["dro"] for precision, model in compiled_cpu_models.items()}

accuracy_rows = []
for precision, output in openvino_cpu_outputs.items():
    metrics = comparison_metrics(pytorch_output, output)
    absolute_error = np.abs(pytorch_output - output)
    relative_error = absolute_error / np.maximum(np.abs(pytorch_output), 1e-6)
    metrics["max_relative_error"] = float(relative_error.max())
    metrics["relation_agreement"] = float(np.mean(np.isclose(pytorch_output, output, rtol=5e-3, atol=5e-4)))
    accuracy_rows.append({"Precision": precision, **metrics})

accuracy_table = pd.DataFrame(accuracy_rows)
display(accuracy_table)

accuracy_by_precision = accuracy_table.set_index("Precision")
assert accuracy_by_precision.loc["FP32", "max_absolute_error"] < 5e-4
assert accuracy_by_precision.loc["FP16", "max_absolute_error"] < 5e-3
assert (accuracy_by_precision["cosine_similarity"] > 0.999).all()
assert (accuracy_by_precision["relation_agreement"] > 0.999).all()

The next cell verifies that both dynamic OpenVINO model variants support DRO-Grasp's published 512-point input shape and preserve numerical accuracy against the PyTorch reference.

In [ ]:
PUBLISHED_NUM_POINTS = 512
torch.manual_seed(42)
published_inputs = (
    torch.randn(1, PUBLISHED_NUM_POINTS, 3, dtype=torch.float32) * POINT_CLOUD_SCALE,
    torch.randn(1, PUBLISHED_NUM_POINTS, 3, dtype=torch.float32) * POINT_CLOUD_SCALE,
    torch.randn(1, config.latent_dim, dtype=torch.float32),
)

with torch.no_grad():
    published_pytorch_output = pytorch_model(*published_inputs).numpy()

published_input_data = {
    "robot_pc": published_inputs[0].numpy(),
    "object_pc": published_inputs[1].numpy(),
    "latent": published_inputs[2].numpy(),
}
published_openvino_outputs = {precision: model(published_input_data)["dro"] for precision, model in compiled_cpu_models.items()}
published_accuracy_rows = [
    {"Precision": precision, **comparison_metrics(published_pytorch_output, output)} for precision, output in published_openvino_outputs.items()
]
published_accuracy_table = pd.DataFrame(published_accuracy_rows)

for precision, output in published_openvino_outputs.items():
    assert output.shape == (1, PUBLISHED_NUM_POINTS, PUBLISHED_NUM_POINTS)

published_accuracy_by_precision = published_accuracy_table.set_index("Precision")
assert published_accuracy_by_precision.loc["FP32", "max_absolute_error"] < 1e-3
assert published_accuracy_by_precision.loc["FP16", "max_absolute_error"] < 5e-3
assert (published_accuracy_by_precision["cosine_similarity"] > 0.999).all()
print(f"Published-shape outputs: {[output.shape for output in published_openvino_outputs.values()]}")
display(published_accuracy_table)

## Run inference on Intel CPU and GPU
[back to top](#Table-of-contents:)

OpenVINO discovers installed devices at runtime. Select any available device for interactive inference with the FP16 model. The explicit benchmark below tests both FP32 and FP16 on CPU and additionally on Intel GPU when available.

In [ ]:
from notebook_utils import device_widget

device = device_widget(default="AUTO", exclude=["NPU"])
device

In [ ]:
selected_precision = "FP16"
selected_model = core.compile_model(ov_models[selected_precision], device.value, {"PERFORMANCE_HINT": "LATENCY"})
selected_output = selected_model(input_data)["dro"]
assert selected_output.shape == (1, NUM_POINTS, NUM_POINTS)
print(f"{selected_precision} DRO output shape on {device.value}: {selected_output.shape}")

The output is a dense robot-to-object distance-relation matrix. A simple inspection plots it as a heatmap; downstream DRO-Grasp optimization consumes this matrix to produce robot joint configurations.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 5))
plt.imshow(selected_output[0], cmap="viridis")
plt.colorbar(label="Predicted distance relation")
plt.xlabel("Object point index")
plt.ylabel("Robot point index")
plt.title(f"DRO-Grasp {selected_precision} output on {device.value}")
plt.show()

## Benchmark PyTorch and OpenVINO
[back to top](#Table-of-contents:)

The benchmark uses three warm-up iterations followed by 20 synchronous timed iterations with identical inputs. It records model compilation time, mean/median/p95 latency, and single-sample throughput for PyTorch FP32 and both OpenVINO FP32 and FP16 on each available device. Results are host-specific and should be regenerated on the target system.

In [ ]:
import time

BENCHMARK_ITERATIONS = 20
benchmark_rows = []

with torch.no_grad():
    pytorch_performance = benchmark(
        lambda: pytorch_model(robot_pc, object_pc, latent),
        iterations=BENCHMARK_ITERATIONS,
    )
benchmark_rows.append(
    {
        "Runtime": "PyTorch",
        "Precision": "FP32",
        "Device": "CPU",
        "Compile time (ms)": 0.0,
        **pytorch_performance,
    }
)

openvino_outputs = {}
for precision, model in ov_models.items():
    for target_device in available_inference_devices(core):
        try:
            compile_start = time.perf_counter()
            compiled_model = core.compile_model(model, target_device, {"PERFORMANCE_HINT": "LATENCY"})
            compile_time_ms = (time.perf_counter() - compile_start) * 1000
            result_key = (precision, target_device)
            openvino_outputs[result_key] = compiled_model(input_data)["dro"]
            performance = benchmark(
                lambda runtime_model=compiled_model: runtime_model(input_data),
                iterations=BENCHMARK_ITERATIONS,
            )
            benchmark_rows.append(
                {
                    "Runtime": "OpenVINO",
                    "Precision": precision,
                    "Device": target_device,
                    "Compile time (ms)": compile_time_ms,
                    **performance,
                }
            )
        except RuntimeError as error:
            print(f"Skipping OpenVINO {precision} on {target_device}: {error}")

benchmark_table = pd.DataFrame(benchmark_rows).rename(
    columns={
        "mean_latency_ms": "Mean latency (ms)",
        "median_latency_ms": "Median latency (ms)",
        "p95_latency_ms": "P95 latency (ms)",
        "throughput_fps": "Throughput (samples/s)",
    }
)
display(benchmark_table.round(3))

## Analyze accuracy and performance
[back to top](#Table-of-contents:)

The following tables compare every available OpenVINO precision/device combination with the same PyTorch FP32 output and calculate speedup relative to PyTorch CPU. A bar chart summarizes median latency.

In [ ]:
accuracy_rows = []
for (precision, target_device), target_output in openvino_outputs.items():
    device_metrics = comparison_metrics(pytorch_output, target_output)
    accuracy_rows.append({"Precision": precision, "Device": target_device, **device_metrics})

accuracy_by_device = pd.DataFrame(accuracy_rows).rename(
    columns={
        "max_absolute_error": "Max absolute error",
        "mean_absolute_error": "Mean absolute error",
        "cosine_similarity": "Cosine similarity",
    }
)

torch_latency = benchmark_table.loc[benchmark_table["Runtime"] == "PyTorch", "Median latency (ms)"].iloc[0]
benchmark_table["Speedup vs PyTorch CPU"] = torch_latency / benchmark_table["Median latency (ms)"]
benchmark_table["Configuration"] = benchmark_table["Runtime"] + " " + benchmark_table["Precision"] + " " + benchmark_table["Device"]

display(accuracy_by_device)
display(benchmark_table.drop(columns="Configuration").round(3))

benchmark_table.plot.bar(
    x="Configuration",
    y="Median latency (ms)",
    legend=False,
    color=["#0071c5", "#00a65a", "#f2b134", "#d94735", "#6f5aa8"][: len(benchmark_table)],
)
plt.ylabel("Median latency (ms)")
plt.title(f"DRO-Grasp latency ({NUM_POINTS} points)")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

## Reusable inference utilities
[back to top](#Table-of-contents:)

`dro_grasp_helper.py` contains the deterministic PyTorch wrapper, stable OpenVINO tensor naming, input validation, device discovery, accuracy metrics, and synchronized benchmark utility. `OpenVINODROGrasp` packages compilation and synchronous inference for reuse in applications.

The next cell is an optional API usage example that shows how application code can run the converted model through `OpenVINODROGrasp`. It is not required for model conversion, accuracy validation, or benchmarking.

In [ ]:
reusable_precision = "FP16"
reusable_model = OpenVINODROGrasp(core, ov_models[reusable_precision], device="CPU")
reusable_output = reusable_model(
    robot_pc=input_data["robot_pc"],
    object_pc=input_data["object_pc"],
    latent=input_data["latent"],
)
assert reusable_output.shape == (1, NUM_POINTS, NUM_POINTS)
print(f"Reusable {reusable_precision} runtime output: {reusable_output.shape}")

## Notebook validation and tests
[back to top](#Table-of-contents:)

The assertions below cover downloaded and generated FP32/FP16 model files, input/output contracts, quick and published point counts, numerical parity, CPU execution, optional GPU execution, and benchmark integrity. Repository checks can be run from the repository root:

```bash
python .ci/check_notebooks.py
nbqa black notebooks/dro-grasp/dro-grasp.ipynb --check --line-length=160
treon notebooks/dro-grasp/dro-grasp.ipynb
```

In [ ]:
assert CHECKPOINT_PATH.is_file()
assert all(model_path.is_file() for model_path in IR_PATHS.values())
assert set(input_data) == {"robot_pc", "object_pc", "latent"}
assert all(output.shape == pytorch_output.shape for output in openvino_cpu_outputs.values())
assert all(output.shape == published_pytorch_output.shape for output in published_openvino_outputs.values())
assert {("FP32", "CPU"), ("FP16", "CPU")} <= set(openvino_outputs)
assert np.isfinite(benchmark_table["Median latency (ms)"]).all()
assert (benchmark_table["Median latency (ms)"] > 0).all()

if "GPU" in available_inference_devices(core):
    assert {("FP32", "GPU"), ("FP16", "GPU")} <= set(openvino_outputs)
    for precision in IR_PATHS:
        assert comparison_metrics(pytorch_output, openvino_outputs[(precision, "GPU")])["cosine_similarity"] > 0.999

print("All DRO-Grasp FP32/FP16 notebook validation checks passed.")

## Conclusions
[back to top](#Table-of-contents:)

DRO-Grasp's deterministic PyTorch validation graph converts directly to explicit FP32 and FP16 OpenVINO IR variants when the latent tensor is made an explicit input and the tensor-only `dro` output is exposed. Both dynamic IR variants were validated with 32-point and published 512-point inputs and runs on Intel CPU and GPU through OpenVINO.

The numerical comparisons in this notebook measure conversion fidelity. Reproducing task-level grasp success additionally requires the DRO-Grasp datasets, robot assets, optimization pipeline, and CUDA-only Isaac Gym evaluation environment.